# Phase 1 – National Noise Exposure Classification

Extract BTS noise levels at each school and assign WHO tiers.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.noise_classification import run_phase1, national_summary
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

## Run Phase 1 Pipeline

In [ ]:
schools = run_phase1()
print(f'Total schools: {len(schools):,}')
schools.head()

## Noise Tier Distribution

In [ ]:
tier_counts = schools['noise_tier_label'].value_counts()
print(tier_counts)
n3_4 = tier_counts.get('Tier 3 - Significant Impact', 0) + tier_counts.get('Tier 4 - Severe Exposure', 0)
print(f'\n% in Tier 3+4: {n3_4/len(schools)*100:.1f}%')

In [ ]:
COLORS = {
    'Tier 1 - Minimal Risk': '#2ecc71',
    'Tier 2 - Elevated Risk': '#f1c40f',
    'Tier 3 - Significant Impact': '#e67e22',
    'Tier 4 - Severe Exposure': '#e74c3c',
}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = [COLORS.get(k, '#888') for k in tier_counts.index]
tier_counts.plot(kind='bar', ax=ax1, color=colors, edgecolor='white', linewidth=0.5)
ax1.set_title('Schools by WHO Noise Tier')
ax1.set_ylabel('Number of Schools')
ax1.tick_params(axis='x', rotation=30)

if 'noise_db' in schools.columns:
    schools['noise_db'].dropna().plot(kind='hist', bins=50, ax=ax2, color='#3498db', edgecolor='white', linewidth=0.3)
    for thresh, color in [(50, '#f1c40f'), (55, '#e67e22'), (65, '#e74c3c')]:
        ax2.axvline(thresh, color=color, linewidth=2, linestyle='--', label=f'{thresh} dB')
    ax2.set_title('Noise Level Distribution')
    ax2.set_xlabel('dB LAeq')
    ax2.legend()

plt.tight_layout()
plt.show()

## State-by-State Summary

In [ ]:
summary = national_summary(schools)
summary.head(15)

In [ ]:
if not summary.empty and 'pct_high_noise' in summary.columns:
    top15 = summary.head(15)[['pct_high_noise']]
    top15.plot(kind='barh', figsize=(10, 6), color='#e74c3c', legend=False)
    plt.xlabel('% Schools in Tier 3 or 4')
    plt.title('Top 15 States: Elementary Schools in High Noise Zones')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()